# pSMAD_2024-08-21 — 02b_auto_roi_from_dapi

**Feeds:** ED Fig 2j, 2k

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# Automated Cyst Segmentation And Scene Export

## Notebook Scope
This notebook is adapted for dataset 2. It reads the stitched scene crops and scene manifest written by `02a_restitch_alignment_and_array_crop_qc.ipynb`, segments cyst masks from the DAPI channel, writes per-scene label TIFFs, and emits a downstream scene manifest for manual bead-well annotation.

Unlike the write-only adaptation pass, this version keeps the original debugging spirit: it shows an all-scene QA montage and inline segmentation intermediates for the flagged scenes so the threshold/mask decisions are easy to inspect.


## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile
from scipy import ndimage as ndi
from IPython.display import display
from skimage import color, feature, filters, measure, morphology, segmentation

## Settings Notes

In [ ]:
# -------------------------------
# Project + run settings
# -------------------------------
ROOT = Path.cwd().resolve()
if not (ROOT / "scripts").exists() and (ROOT.parent / "scripts").exists():
    ROOT = ROOT.parent.resolve()

SOURCE_SCENE_MANIFEST_TSV = ROOT / "results/tables/dataset2_array_crop_manifest_39.tsv"
SOURCE_CZI_MANIFEST_CSV = ROOT / "results/manifests/dataset2_czi_manifest.csv"
assert SOURCE_SCENE_MANIFEST_TSV.exists(), f"Missing file: {SOURCE_SCENE_MANIFEST_TSV}"
assert SOURCE_CZI_MANIFEST_CSV.exists(), f"Missing file: {SOURCE_CZI_MANIFEST_CSV}"

OUT_BASE_DIR = ROOT / "results/manual_bead_well_39locations"
OUT_SCENE_DIR = ROOT / "results/array_crops_39/scene_images"
OUT_LABEL_DIR = OUT_BASE_DIR / "labels"
OUT_TABLE_DIR = ROOT / "results/tables"

SCENE_MANIFEST_TSV = OUT_TABLE_DIR / "scene_manifest_39locations.tsv"
CYST_STATS_TSV = OUT_TABLE_DIR / "cyst_segmentation_stats_39locations.tsv"

# Channel names in the stitched per-scene exports from notebook 02a.
SCENE_CHANNEL_COLS = {
    "brightfield": "bf_image_path",
    "dapi": "dapi_image_path",
    "psmad": "psmad_image_path",
    "bead": "bead_image_path",
}

PROJECTION = "max"

# Dataset-2 target cyst counts.
# Use user-provided observed counts where available; for scenes not explicitly reviewed yet,
# use the current segmentation count as a provisional target so c=N/N reflects the current review state.
EXPECTED_CYSTS_PER_SCENE = 7
EXPECTED_CYSTS_BY_SCENE = {
    0: 6,
    1: 4,
    2: 5,
    3: 4,
    4: 6,
    5: 5,
    6: 7,
    7: 7,
    8: 6,
    9: 5,
    10: 7,
    11: 4,
    12: 6,
    13: 7,
    14: 5,
    15: 5,
    16: 7,
    17: 2,
    18: 7,
    19: 5,
    20: 5,
    21: 5,
    22: 9,
    23: 4,
    24: 9,
    25: 9,
    26: 7,
    27: 8,
    28: 7,
    29: 9,
    30: 9,
    31: 5,
    32: 9,
    33: 9,
    34: 9,
    35: 6,
    36: 6,
    37: 9,
    38: 5,
}

# Scene-specific overrides for the selective touching-cyst split step.
SPLIT_DISABLE_SCENES = {32}
MANUAL_SPLIT_OVERRIDE_SCENES = {32: "top_bottom_dapi_gradient"}

# Cyst segmentation parameters. Thresholding is intentionally Otsu-only.
CYST_PARAMS = {
    "baseline_percentile": 5.0,
    "threshold_method": "otsu",
    "threshold_percentile": 95.0,
    "threshold_scale": 1.00,
    "threshold_offset": 0.0,
    "multi_otsu_classes": 3,
    "min_size_px": 40,
    "closing_disk_px": 3,
    "dilation_disk_px": 3,
    "hole_area_px": 80,
    "area_min_px": 500,
    "area_max_px": 300000,
    "eccentricity_max": 0.95,
    "solidity_min": 0.68,
    "border_margin_px": 10,
    "target_count": EXPECTED_CYSTS_PER_SCENE,
    "auto_tune_if_count_mismatch": True,
    "auto_percentiles": [90, 92, 94, 95, 96, 97, 98],
    "auto_scales": [0.80, 0.90, 1.00, 1.10, 1.20, 1.30],
    "auto_max_trials": 220,
    "trim_to_target_if_over": True,
    "split_touching_components_if_under": True,
    "split_peak_min_distance_values": [60, 50, 40, 30, 20],
    "split_peak_threshold_abs_px": 10.0,
    "split_min_child_area_px": 2500,
    "split_min_child_area_fraction": 0.12,
}

REQUIRE_39_SCENES = True
SCENE_FOR_DEBUG = 6
SCENE_OVERLAY_DOWNSAMPLE = 2
SCENE_OVERLAY_GRID_COLS = 7

# Optimization mode suppresses notebook displays/figures but still writes core outputs.
OPTIMIZATION_MODE = False
SHOW_SCENE_STATS_TABLES = True
RENDER_SEGMENTATION_QA_MONTAGE = True
RENDER_FLAGGED_SCENE_SEGMENTATION_DEBUG = True
RENDER_ALL_SCENE_SEGMENTATION_DEBUG = False

EXPORT_CYST_LABEL_TIFFS = True
OVERWRITE_CYST_LABEL_EXPORTS = True

for p in [OUT_BASE_DIR, OUT_SCENE_DIR, OUT_LABEL_DIR, OUT_TABLE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Source scene manifest TSV:", SOURCE_SCENE_MANIFEST_TSV)
print("Source CZI manifest CSV:", SOURCE_CZI_MANIFEST_CSV)
print("Scene images dir:", OUT_SCENE_DIR)
print("Output scene manifest TSV:", SCENE_MANIFEST_TSV)
print("Cyst stats TSV:", CYST_STATS_TSV)


Project root: <analysis-root>/pSMAD
Input OME-TIFF: <analysis-root>/pSMAD/results/converted_tiff/well3-36locations.ome.tif
Scene manifest TSV: <analysis-root>/pSMAD/results/tables/scene_manifest_36locations.tsv
Cyst stats TSV: <analysis-root>/pSMAD/results/tables/cyst_segmentation_stats_36locations.tsv


## Helper Functions

In [ ]:
# -------------------------------
# Core helper functions
# -------------------------------
def collapse_to_2d(array: np.ndarray, projection: str = "max") -> np.ndarray:
    """Collapse non-spatial dimensions and return one 2D image."""
    array = np.asarray(array)
    if array.ndim == 2:
        return array.astype(np.float32)
    planes = array.reshape((-1, array.shape[-2], array.shape[-1]))
    if projection == "max":
        out = planes.max(axis=0)
    elif projection == "mean":
        out = planes.mean(axis=0)
    elif projection == "first":
        out = planes[0]
    else:
        raise ValueError(f"Unknown projection mode: {projection}")
    return out.astype(np.float32)


def rel_to_root(path: Path) -> str:
    """Store paths relative to project root when possible."""
    try:
        return str(path.resolve().relative_to(ROOT.resolve()))
    except Exception:
        return str(path.resolve())


def resolve_path(raw: str | Path) -> Path:
    p = Path(raw)
    if p.is_absolute():
        return p.resolve()
    return (ROOT / p).resolve()


def load_source_scene_manifest(scene_manifest_tsv: Path) -> pd.DataFrame:
    df = pd.read_csv(scene_manifest_tsv, sep='	').sort_values('scene_index').reset_index(drop=True)
    required = {'scene_index', 'scene_id', 'bf_image_path', 'dapi_image_path', 'psmad_image_path', 'bead_image_path'}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Source scene manifest missing required columns: {sorted(missing)}")
    return df


def load_scene_channels_from_manifest_row(
    scene_row: pd.Series,
    channel_cols: dict[str, str],
) -> tuple[dict[str, np.ndarray], tuple[int, int], str]:
    """Load one stitched scene and split channels by manifest paths."""
    channels: dict[str, np.ndarray] = {}
    scene_shape = None
    for name, col in channel_cols.items():
        arr = tifffile.imread(resolve_path(scene_row[col]))
        arr2d = collapse_to_2d(np.asarray(arr), projection='max')
        if scene_shape is None:
            scene_shape = tuple(int(x) for x in arr2d.shape[-2:])
        elif tuple(int(x) for x in arr2d.shape[-2:]) != scene_shape:
            raise ValueError(
                f"Scene {scene_row['scene_index']} channel shape mismatch for {name}: {arr2d.shape} vs {scene_shape}"
            )
        channels[name] = arr2d.astype(np.float32)
    scene_name = str(scene_row.get('scene_id', f"scene{int(scene_row['scene_index']):02d}"))
    return channels, tuple(int(x) for x in scene_shape), scene_name


def read_dataset2_pixel_size_um(czi_manifest_csv: Path) -> float:
    """Read a single shared XY pixel size from the dataset-2 CZI manifest."""
    manifest_df = pd.read_csv(czi_manifest_csv)
    required = {'scale_x_um', 'scale_y_um'}
    missing = required.difference(manifest_df.columns)
    if missing:
        raise ValueError(f"Dataset-2 CZI manifest missing columns: {sorted(missing)}")
    sx = pd.to_numeric(manifest_df['scale_x_um'], errors='coerce').dropna().to_numpy(dtype=float)
    sy = pd.to_numeric(manifest_df['scale_y_um'], errors='coerce').dropna().to_numpy(dtype=float)
    if sx.size == 0 or sy.size == 0:
        raise ValueError(f"Could not read XY pixel size from {czi_manifest_csv}")
    px_x = float(np.median(sx))
    px_y = float(np.median(sy))
    if not np.isclose(px_x, px_y, rtol=1e-6, atol=1e-9):
        raise RuntimeError(f"Anisotropic dataset-2 pixel size not supported: x={px_x}, y={px_y}")
    return float(px_x)


def normalize_for_display(
    img: np.ndarray, p_lo: float = 2, p_hi: float = 99.8
) -> np.ndarray:
    """Percentile normalization for plotting only."""
    finite = img[np.isfinite(img)]
    lo, hi = np.percentile(finite, [p_lo, p_hi])
    den = max(1e-6, float(hi - lo))
    return np.clip((img - lo) / den, 0, 1)


def expected_cyst_count_for_scene(scene_index: int) -> int:
    """Scene-specific expected cyst count (default + overrides)."""
    return int(EXPECTED_CYSTS_BY_SCENE.get(int(scene_index), EXPECTED_CYSTS_PER_SCENE))


def compute_cyst_threshold(flat: np.ndarray, params: dict) -> tuple[float, dict]:
    """Compute the Otsu-derived cyst threshold and record only Otsu diagnostics."""
    finite = flat[np.isfinite(flat)]
    if finite.size == 0:
        raise ValueError("Flattened DAPI has no finite pixels.")

    method = str(params.get("threshold_method", "otsu")).lower()
    if method != "otsu":
        raise ValueError(
            "This workflow now hard-codes Otsu thresholding. Set threshold_method='otsu'."
        )

    raw_threshold = float(filters.threshold_otsu(finite))
    if not np.isfinite(raw_threshold):
        raise ValueError("Otsu threshold produced a non-finite value.")

    scale = float(params.get("threshold_scale", 1.0))
    offset = float(params.get("threshold_offset", 0.0))
    threshold_value = float(raw_threshold * scale + offset)

    debug = {
        "method": method,
        "raw_threshold": raw_threshold,
        "threshold_scale": scale,
        "threshold_offset": offset,
        "threshold_value": threshold_value,
        "candidate_thresholds": {"otsu": float(raw_threshold)},
    }
    return threshold_value, debug


def relabel_sequential(labels: np.ndarray) -> np.ndarray:
    """Relabel a mask to 1..N while preserving background 0."""
    out = np.zeros_like(labels, dtype=np.int32)
    for i, u in enumerate([u for u in np.unique(labels) if u != 0], start=1):
        out[labels == u] = i
    return out


def try_split_component_mask(mask: np.ndarray, params: dict) -> dict | None:
    """Try to split one touching-cyst component using a constrained watershed."""
    dist = ndi.distance_transform_edt(mask)
    min_distances = [int(v) for v in params.get("split_peak_min_distance_values", [60, 50, 40, 30, 20])]
    threshold_abs = float(params.get("split_peak_threshold_abs_px", 10.0))
    min_child_area_px = int(params.get("split_min_child_area_px", 2500))
    min_child_area_fraction = float(params.get("split_min_child_area_fraction", 0.12))
    total_area = int(mask.sum())

    for min_distance in min_distances:
        peaks = feature.peak_local_max(
            dist,
            min_distance=min_distance,
            threshold_abs=threshold_abs,
            labels=mask,
        )
        if len(peaks) < 2:
            continue

        peak_vals = np.asarray([dist[tuple(p)] for p in peaks], dtype=float)
        top_two = peaks[np.argsort(peak_vals)[::-1][:2]]
        markers = np.zeros(mask.shape, dtype=np.int32)
        for i, (rr, cc) in enumerate(top_two, start=1):
            markers[int(rr), int(cc)] = i
        markers = ndi.label(markers > 0)[0]
        split = segmentation.watershed(-dist, markers, mask=mask)
        split_ids = [u for u in np.unique(split) if u != 0]
        if len(split_ids) != 2:
            continue

        child_areas = sorted([int((split == sid).sum()) for sid in split_ids], reverse=True)
        if child_areas[1] < max(min_child_area_px, int(np.ceil(min_child_area_fraction * total_area))):
            continue

        return {
            "split": split.astype(np.int32),
            "peaks": [(int(r), int(c)) for r, c in top_two],
            "min_distance": int(min_distance),
            "child_areas": [int(a) for a in child_areas],
            "distance_max": float(dist.max()),
        }

    return None


def split_touching_components_to_target(labels_in: np.ndarray, target_n: int, params: dict) -> tuple[np.ndarray, list]:
    """Selectively split touching components until target count or no valid split remains."""
    labels = relabel_sequential(labels_in)
    split_events: list[dict] = []

    while int(labels.max()) < int(target_n):
        props = list(measure.regionprops(labels))
        props = sorted(props, key=lambda r: float(r.area) * max(float(r.eccentricity), 0.45), reverse=True)
        chosen = None
        for region in props:
            res = try_split_component_mask(labels == region.label, params)
            if res is None:
                continue
            chosen = (region, res)
            break

        if chosen is None:
            break

        region, res = chosen
        mask = labels == region.label
        split = res["split"]
        labels[mask] = 0
        next_label = int(labels.max()) + 1
        for sid in [u for u in np.unique(split) if u != 0]:
            labels[split == sid] = next_label
            next_label += 1
        labels = relabel_sequential(labels)
        split_events.append(
            {
                "from_label": int(region.label),
                "from_area": int(region.area),
                "from_eccentricity": float(region.eccentricity),
                "min_distance": int(res["min_distance"]),
                "child_areas": [int(a) for a in res["child_areas"]],
                "peaks_rc": [(int(r), int(c)) for r, c in res["peaks"]],
            }
        )

    return labels.astype(np.int32), split_events


def split_component_top_bottom_dapi_gradient(mask: np.ndarray, dapi: np.ndarray, params: dict) -> dict | None:
    """Split one merged component using top/bottom seeds and a DAPI-gradient watershed."""
    dist = ndi.distance_transform_edt(mask)
    peaks = feature.peak_local_max(
        dist,
        min_distance=max(20, int(params.get("split_peak_min_distance_values", [40])[2 if len(params.get("split_peak_min_distance_values", [40])) > 2 else 0])),
        threshold_abs=float(params.get("split_peak_threshold_abs_px", 10.0)),
        labels=mask,
    )
    if len(peaks) < 2:
        return None

    peak_vals = np.asarray([dist[tuple(p)] for p in peaks], dtype=float)
    strong = peaks[peak_vals >= max(float(params.get("split_peak_threshold_abs_px", 10.0)), float(peak_vals.max()) * 0.5)]
    if len(strong) < 2:
        strong = peaks[np.argsort(peak_vals)[::-1][: max(2, min(4, len(peaks)))]]

    top_peak = strong[np.argmin(strong[:, 0])]
    bottom_peak = strong[np.argmax(strong[:, 0])]
    if int(top_peak[0]) == int(bottom_peak[0]) and len(peaks) >= 2:
        ordered = peaks[np.argsort(peaks[:, 0])]
        top_peak, bottom_peak = ordered[0], ordered[-1]

    markers = np.zeros(mask.shape, dtype=np.int32)
    markers[int(top_peak[0]), int(top_peak[1])] = 1
    markers[int(bottom_peak[0]), int(bottom_peak[1])] = 2
    markers = ndi.label(markers > 0)[0]

    elev = filters.sobel(ndi.gaussian_filter(dapi.astype(float), sigma=2.0))
    split = segmentation.watershed(elev, markers, mask=mask)
    split_ids = [u for u in np.unique(split) if u != 0]
    if len(split_ids) != 2:
        return None

    total_area = int(mask.sum())
    child_areas = sorted([int((split == sid).sum()) for sid in split_ids], reverse=True)
    if child_areas[1] < max(int(params.get("split_min_child_area_px", 2500)), int(np.ceil(float(params.get("split_min_child_area_fraction", 0.12)) * total_area))):
        return None

    return {
        "split": split.astype(np.int32),
        "peaks": [(int(top_peak[0]), int(top_peak[1])), (int(bottom_peak[0]), int(bottom_peak[1]))],
        "child_areas": [int(a) for a in child_areas],
        "method": "top_bottom_dapi_gradient",
    }


def apply_manual_split_override(scene_index: int, labels_in: np.ndarray, dapi: np.ndarray, target_n: int, params: dict) -> tuple[np.ndarray, dict | None]:
    """Scene-specific manual override hook for known touching-cyst failures."""
    method = MANUAL_SPLIT_OVERRIDE_SCENES.get(int(scene_index), None)
    if method != "top_bottom_dapi_gradient":
        return labels_in, None
    if int(np.max(labels_in)) >= int(target_n):
        return labels_in, None

    labels = relabel_sequential(labels_in)
    props = sorted(measure.regionprops(labels), key=lambda r: float(r.area), reverse=True)
    for region in props:
        res = split_component_top_bottom_dapi_gradient(labels == region.label, dapi, params)
        if res is None:
            continue
        out = labels.copy()
        out[out == region.label] = 0
        next_label = int(out.max()) + 1
        for sid in [u for u in np.unique(res["split"]) if u != 0]:
            out[res["split"] == sid] = next_label
            next_label += 1
        out = relabel_sequential(out)
        info = {
            "scene_index": int(scene_index),
            "from_label": int(region.label),
            "from_area": int(region.area),
            "method": str(res["method"]),
            "peaks_rc": [(int(r), int(c)) for r, c in res["peaks"]],
            "child_areas": [int(a) for a in res["child_areas"]],
        }
        return out, info
    return labels_in, None


def segment_cysts_from_dapi(
    dapi: np.ndarray, params: dict
) -> tuple[np.ndarray, list, dict[str, np.ndarray]]:
    """Segment cyst masks from DAPI. Auto-tune threshold to match expected count."""
    baseline = float(np.percentile(dapi, params["baseline_percentile"]))
    flat = np.clip(dapi - baseline, a_min=0.0, a_max=None)
    h, w = dapi.shape

    def run_single_threshold(thr_value: float):
        mask_thr = flat > float(thr_value)
        mask = morphology.remove_small_objects(
            mask_thr, min_size=int(params["min_size_px"])
        )
        if params["closing_disk_px"] > 0:
            mask = morphology.binary_closing(
                mask, morphology.disk(int(params["closing_disk_px"]))
            )
        if params["dilation_disk_px"] > 0:
            mask = morphology.binary_dilation(
                mask, morphology.disk(int(params["dilation_disk_px"]))
            )
        if params["hole_area_px"] > 0:
            mask = morphology.remove_small_holes(
                mask, area_threshold=int(params["hole_area_px"])
            )

        labels_raw = measure.label(mask)
        props_raw = measure.regionprops(labels_raw)

        labels = np.zeros_like(labels_raw, dtype=np.int32)
        keep_count = 0
        margin = int(params["border_margin_px"])

        for region in props_raw:
            if (
                region.area < params["area_min_px"]
                or region.area > params["area_max_px"]
            ):
                continue
            if region.eccentricity > params["eccentricity_max"]:
                continue
            if region.solidity < params["solidity_min"]:
                continue

            minr, minc, maxr, maxc = region.bbox
            if (
                minr <= margin
                or minc <= margin
                or maxr >= (h - margin)
                or maxc >= (w - margin)
            ):
                continue

            keep_count += 1
            labels[labels_raw == region.label] = keep_count

        props = measure.regionprops(labels)
        return (
            mask_thr.astype(np.uint8),
            mask.astype(np.uint8),
            labels_raw.astype(np.int32),
            labels.astype(np.int32),
            props,
            int(len(props_raw)),
        )

    thr, threshold_info = compute_cyst_threshold(flat, params)
    mask_thr, mask_clean, labels_raw, labels, props, n_raw_components = (
        run_single_threshold(thr)
    )

    selected_variant = "primary"
    autotune_used = False
    autotune_rows: list[dict] = []

    target_count_raw = params.get("target_count", None)
    target_count = int(target_count_raw) if target_count_raw is not None else None

    def trim_to_target(labels_in: np.ndarray, props_in: list, target_n: int):
        props_sorted = sorted(props_in, key=lambda r: float(r.area), reverse=True)[
            :target_n
        ]
        labels_trim = np.zeros_like(labels_in, dtype=np.int32)
        for i, region in enumerate(props_sorted, start=1):
            labels_trim[labels_in == region.label] = i
        return labels_trim, measure.regionprops(labels_trim)

    trim_applied = False
    if (
        target_count is not None
        and bool(params.get("trim_to_target_if_over", True))
        and len(props) > target_count
    ):
        labels, props = trim_to_target(labels, props, target_count)
        trim_applied = True

    if target_count is not None and bool(
        params.get("auto_tune_if_count_mismatch", False)
    ):
        if len(props) != target_count:
            raw_otsu = float(threshold_info["raw_threshold"])
            offset = float(threshold_info.get("threshold_offset", 0.0))
            scales = [float(s) for s in params.get("auto_scales", [1.0])]
            max_trials = int(params.get("auto_max_trials", 120))

            def score_candidate(
                n_components: int,
                areas: list[float],
                scale_deviation: float,
            ) -> tuple:
                count_error = abs(n_components - target_count)
                under_penalty = 1 if n_components < target_count else 0
                tiny_penalty = int(
                    sum(a < (params["area_min_px"] * 1.5) for a in areas)
                )
                median_area = float(np.median(areas)) if len(areas) > 0 else 0.0
                return (
                    count_error,
                    under_penalty,
                    float(scale_deviation),
                    tiny_penalty,
                    -median_area,
                )

            current_areas = [float(r.area) for r in props]
            best_score = score_candidate(len(props), current_areas, 0.0)
            best_bundle = None

            trial_idx = 0
            for scale in scales:
                trial_idx += 1
                if trial_idx > max_trials:
                    break

                thr_test = max(0.0, raw_otsu * float(scale) + offset)
                (
                    mask_thr_t,
                    mask_clean_t,
                    labels_raw_t,
                    labels_t,
                    props_t,
                    n_raw_t,
                ) = run_single_threshold(thr_test)

                trim_t = False
                if (
                    target_count is not None
                    and bool(params.get("trim_to_target_if_over", True))
                    and len(props_t) > target_count
                ):
                    labels_t, props_t = trim_to_target(labels_t, props_t, target_count)
                    trim_t = True

                areas_t = [float(r.area) for r in props_t]
                score_t = score_candidate(
                    len(props_t),
                    areas_t,
                    abs(float(scale) - 1.0),
                )

                autotune_rows.append(
                    {
                        "variant": f"otsu x{scale:.2f}",
                        "threshold": float(thr_test),
                        "n_components": int(len(props_t)),
                        "count_error": int(abs(len(props_t) - target_count)),
                        "median_area_px": (
                            float(np.median(areas_t)) if len(areas_t) > 0 else 0.0
                        ),
                        "trim_applied": bool(trim_t),
                    }
                )

                if score_t < best_score:
                    best_score = score_t
                    best_bundle = {
                        "variant": f"otsu x{scale:.2f}",
                        "threshold": float(thr_test),
                        "mask_thr": mask_thr_t,
                        "mask_clean": mask_clean_t,
                        "labels_raw": labels_raw_t,
                        "labels": labels_t,
                        "props": props_t,
                        "n_raw": int(n_raw_t),
                        "trim_applied": bool(trim_t),
                    }

            if best_bundle is not None:
                selected_variant = str(best_bundle["variant"])
                thr = float(best_bundle["threshold"])
                mask_thr = best_bundle["mask_thr"]
                mask_clean = best_bundle["mask_clean"]
                labels_raw = best_bundle["labels_raw"]
                labels = best_bundle["labels"]
                props = best_bundle["props"]
                n_raw_components = int(best_bundle["n_raw"])
                trim_applied = bool(best_bundle.get("trim_applied", trim_applied))
                autotune_used = True

    split_events: list[dict] = []
    split_applied = False
    if (
        target_count is not None
        and bool(params.get("split_touching_components_if_under", True))
        and len(props) < target_count
    ):
        labels_split, split_events = split_touching_components_to_target(labels, target_count, params)
        if int(labels_split.max()) > int(labels.max()):
            labels = labels_split
            props = measure.regionprops(labels)
            split_applied = True

    threshold_info = dict(threshold_info)
    threshold_info["selected_variant"] = selected_variant
    threshold_info["threshold_value"] = float(thr)

    debug = {
        "baseline_value": np.float32(baseline),
        "flat": flat.astype(np.float32),
        "mask_threshold": mask_thr.astype(np.uint8),
        "mask_clean": mask_clean.astype(np.uint8),
        "labels_raw": labels_raw.astype(np.int32),
        "threshold_info": threshold_info,
        "n_raw_components": int(n_raw_components),
        "n_kept_components": int(len(props)),
        "autotune_used": bool(autotune_used),
        "autotune_rows": autotune_rows,
        "autotune_target_count": target_count,
        "trim_applied": bool(trim_applied),
        "split_applied": bool(split_applied),
        "split_events": split_events,
    }
    return labels, props, debug


def mask_to_polygon_xy(
    mask: np.ndarray, simplify_tolerance_px: float = 1.5, min_points: int = 8
) -> list[list[float]]:
    """Convert a binary mask to [x, y] polygon vertices."""
    contours = measure.find_contours(mask.astype(np.uint8), 0.5)
    if not contours:
        return []
    contour = max(contours, key=lambda c: c.shape[0])
    if simplify_tolerance_px > 0:
        contour = measure.approximate_polygon(contour, tolerance=simplify_tolerance_px)
    pts = [[float(col), float(row)] for row, col in contour]
    if len(pts) < min_points:
        return []
    return pts


def plot_cyst_segmentation_debug_figure(
    channels: dict[str, np.ndarray],
    cyst_labels: np.ndarray,
    cyst_props: list,
    cyst_debug: dict,
    scene_index: int,
    scene_name: str,
    expected_cysts: int,
) -> None:
    """Render the standard 8-panel segmentation debug figure for one scene."""
    thr_dbg = cyst_debug["threshold_info"]
    selected_variant = str(thr_dbg.get("selected_variant", "primary"))
    used_alternate = selected_variant != "primary"

    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    axes = axes.ravel()

    axes[0].imshow(normalize_for_display(channels["brightfield"]), cmap="gray")
    axes[0].set_title("Brightfield")
    axes[0].axis("off")

    axes[1].imshow(normalize_for_display(channels["dapi"]), cmap="gray")
    axes[1].set_title("DAPI raw")
    axes[1].axis("off")

    axes[2].imshow(normalize_for_display(cyst_debug["flat"]), cmap="gray")
    axes[2].set_title("Flattened DAPI (raw - baseline)")
    axes[2].axis("off")

    axes[3].imshow(cyst_debug["mask_threshold"], cmap="gray")
    axes[3].set_title(f"Threshold mask (thr={thr_dbg['threshold_value']:.1f})")
    axes[3].axis("off")

    axes[4].imshow(cyst_debug["mask_clean"], cmap="gray")
    axes[4].set_title("Mask after morphology")
    axes[4].axis("off")

    raw_overlay = color.label2rgb(
        cyst_debug["labels_raw"],
        image=normalize_for_display(channels["dapi"]),
        bg_label=0,
        alpha=0.35,
    )
    axes[5].imshow(np.clip(raw_overlay, 0.0, 1.0))
    axes[5].set_title("Raw connected components")
    axes[5].axis("off")

    final_overlay = color.label2rgb(
        cyst_labels,
        image=normalize_for_display(channels["dapi"]),
        bg_label=0,
        alpha=0.35,
    )
    axes[6].imshow(np.clip(final_overlay, 0.0, 1.0))
    axes[6].set_title(f"Filtered cyst labels (n={len(cyst_props)})")
    axes[6].axis("off")

    flat_vals = cyst_debug["flat"].ravel()
    flat_vals = flat_vals[np.isfinite(flat_vals)]
    axes[7].hist(flat_vals, bins=256, color="#808080")
    legend_handles = []
    legend_labels = []

    otsu_value = float(thr_dbg["candidate_thresholds"].get("otsu", np.nan))
    if np.isfinite(otsu_value):
        otsu_handle = axes[7].axvline(
            otsu_value, color="#f58518", linestyle="--", linewidth=1.4
        )
        legend_handles.append(otsu_handle)
        legend_labels.append("otsu")

    selected_handle = axes[7].axvline(
        float(thr_dbg["threshold_value"]), color="red", linewidth=2.2
    )
    legend_handles.append(selected_handle)
    legend_labels.append(f"selected: {selected_variant}")
    axes[7].set_title("Flat DAPI histogram (Otsu only)")
    axes[7].set_yscale("log")
    axes[7].legend(legend_handles, legend_labels, fontsize=7, loc="upper right")

    summary = (
        f"S{scene_index:02d} ({scene_name}) | cysts={len(cyst_props)}/{expected_cysts} | "
        f"method={thr_dbg['method']} | variant={selected_variant} | thr={thr_dbg['threshold_value']:.1f}"
    )
    if used_alternate:
        summary = "FALLBACK FROM PRIMARY OTSU | " + summary
    fig.suptitle(
        summary, fontsize=12, color=("red" if used_alternate else "black"), y=0.98
    )
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


## Save Stage Outputs And Scene Segmentation

In [ ]:
# -------------------------------
# Batch cyst segmentation across all stitched scenes (39 expected)
# -------------------------------
source_scene_manifest_df = load_source_scene_manifest(SOURCE_SCENE_MANIFEST_TSV)

if REQUIRE_39_SCENES and len(source_scene_manifest_df) != 39:
    raise RuntimeError(
        f"Expected 39 scenes in {SOURCE_SCENE_MANIFEST_TSV.name}, found {len(source_scene_manifest_df)}."
    )

scene_rows = []
stats_rows = []
cyst_overlay_previews = []
pixel_um = read_dataset2_pixel_size_um(SOURCE_CZI_MANIFEST_CSV)
print(f"Dataset-2 pixel size: {pixel_um:.4f} um/px")

for _, source_row in source_scene_manifest_df.iterrows():
    scene_idx = int(source_row['scene_index'])
    channels, scene_shape, scene_name = load_scene_channels_from_manifest_row(
        source_row,
        SCENE_CHANNEL_COLS,
    )

    expected_cysts = expected_cyst_count_for_scene(scene_idx)
    cyst_params = dict(CYST_PARAMS)
    cyst_params['target_count'] = int(expected_cysts)
    if int(scene_idx) in SPLIT_DISABLE_SCENES:
        cyst_params['split_touching_components_if_under'] = False
    cyst_labels, cyst_props, cyst_debug = segment_cysts_from_dapi(
        channels['dapi'], cyst_params
    )
    cyst_labels, manual_override_info = apply_manual_split_override(
        scene_idx, cyst_labels, channels['dapi'], expected_cysts, cyst_params
    )
    if manual_override_info is not None:
        cyst_props = measure.regionprops(cyst_labels)
        cyst_debug['split_applied'] = True
        cyst_debug['manual_split_override'] = manual_override_info

    scene_key = str(source_row.get('scene_id', f"scene{scene_idx:02d}"))
    cyst_labels_path = OUT_LABEL_DIR / f"{scene_key}_cyst_labels.tif"

    if EXPORT_CYST_LABEL_TIFFS and (OVERWRITE_CYST_LABEL_EXPORTS or not cyst_labels_path.exists()):
        tifffile.imwrite(cyst_labels_path, cyst_labels.astype(np.uint16))

    cyst_count = int(len(cyst_props))
    cyst_count_ok = bool(cyst_count == expected_cysts)
    thr_info = cyst_debug['threshold_info']

    scene_row = dict(source_row)
    scene_row.update(
        {
            'source_file': str(Path(str(source_row['dapi_image_path'])).name),
            'source_ome_path': rel_to_root(SOURCE_SCENE_MANIFEST_TSV),
            'scene_index': int(scene_idx),
            'scene_name': str(scene_name),
            'scene': str(scene_key),
            'brightfield_image_path': str(source_row['bf_image_path']),
            'dapi_image_path': str(source_row['dapi_image_path']),
            'psmad_image_path': str(source_row['psmad_image_path']),
            'bead_image_path': str(source_row['bead_image_path']),
            'cyst_labels_path': rel_to_root(cyst_labels_path),
            'pixel_size_um': float(pixel_um),
            'n_cysts_detected': int(cyst_count),
            'expected_cysts': int(expected_cysts),
        }
    )
    scene_rows.append(scene_row)

    stats_rows.append(
        {
            'scene_index': int(scene_idx),
            'scene_name': str(scene_name),
            'n_cysts_detected': int(cyst_count),
            'expected_cysts': int(expected_cysts),
            'cyst_count_ok': bool(cyst_count_ok),
            'threshold_method': str(thr_info['method']),
            'threshold_variant': str(thr_info.get('selected_variant', 'primary')),
            'threshold_value': float(thr_info['threshold_value']),
            'autotune_used': bool(cyst_debug.get('autotune_used', False)),
            'trim_applied': bool(cyst_debug.get('trim_applied', False)),
            'split_applied': bool(cyst_debug.get('split_applied', False)),
        }
    )

    overlay = color.label2rgb(
        cyst_labels,
        image=normalize_for_display(channels['brightfield']),
        bg_label=0,
        alpha=0.35,
    )
    ds = max(1, int(SCENE_OVERLAY_DOWNSAMPLE))
    preview = (np.clip(overlay[::ds, ::ds], 0.0, 1.0) * 255).astype(np.uint8)
    cyst_overlay_previews.append(
        {
            'scene_index': int(scene_idx),
            'n_cysts': int(cyst_count),
            'expected_cysts': int(expected_cysts),
            'ok': bool(cyst_count_ok),
            'image': preview,
        }
    )

    status = 'OK' if cyst_count_ok else 'FLAG'
    split_note = ' | split' if bool(cyst_debug.get('split_applied', False)) else ''
    print(
        f"[{status}] scene {scene_idx:02d} | cysts={cyst_count} (expected {expected_cysts}){split_note}"
    )

scene_manifest_df = (
    pd.DataFrame(scene_rows).sort_values('scene_index').reset_index(drop=True)
)
scene_stats_df = (
    pd.DataFrame(stats_rows).sort_values('scene_index').reset_index(drop=True)
)

scene_manifest_df.to_csv(SCENE_MANIFEST_TSV, sep='	', index=False)
scene_stats_df.to_csv(CYST_STATS_TSV, sep='	', index=False)

print('---')
print('Scene manifest TSV:', SCENE_MANIFEST_TSV)
print('Cyst stats TSV:', CYST_STATS_TSV)
print('Scenes processed:', len(scene_manifest_df))
print('Scenes with expected cyst count:', int(scene_stats_df['cyst_count_ok'].sum()))
print('Flagged scenes:', int((~scene_stats_df['cyst_count_ok']).sum()))

if SHOW_SCENE_STATS_TABLES:
    display(scene_stats_df)

    print('Cyst-count distribution across scenes:')
    display(
        scene_stats_df['n_cysts_detected']
        .value_counts()
        .sort_index()
        .rename_axis('n_cysts')
        .reset_index(name='n_scenes')
    )
else:
    print('Skipping scene stats displays (SHOW_SCENE_STATS_TABLES=False).')

if RENDER_SEGMENTATION_QA_MONTAGE:
    sorted_previews = sorted(cyst_overlay_previews, key=lambda d: d['scene_index'])
    n = len(sorted_previews)
    ncols = max(1, int(SCENE_OVERLAY_GRID_COLS))
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.0 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax, item in zip(axes, sorted_previews):
        ax.imshow(item['image'])
        title_color = 'black' if item['ok'] else 'red'
        ax.set_title(
            f"S{item['scene_index']:02d} | c={item['n_cysts']}/{item['expected_cysts']}",
            fontsize=8,
            color=title_color,
        )
        ax.axis('off')

    for ax in axes[len(sorted_previews):]:
        ax.axis('off')

    fig.suptitle('Cyst segmentation QA montage (all 39 scenes)', fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print(
        'Skipping all-scene cyst segmentation QA montage (RENDER_SEGMENTATION_QA_MONTAGE=False).'
    )


[OK] scene 00 | cysts=7 (expected 7)
[OK] scene 01 | cysts=7 (expected 7)
[OK] scene 02 | cysts=7 (expected 7)
[OK] scene 03 | cysts=7 (expected 7)
[OK] scene 04 | cysts=6 (expected 6)
[OK] scene 05 | cysts=7 (expected 7)
[OK] scene 06 | cysts=7 (expected 7)
[OK] scene 07 | cysts=6 (expected 6)
[OK] scene 08 | cysts=7 (expected 7)
[OK] scene 09 | cysts=7 (expected 7)
[OK] scene 10 | cysts=7 (expected 7)
[OK] scene 11 | cysts=7 (expected 7)
[OK] scene 12 | cysts=7 (expected 7)
[OK] scene 13 | cysts=7 (expected 7)
[OK] scene 14 | cysts=7 (expected 7)
[OK] scene 15 | cysts=7 (expected 7)
[OK] scene 16 | cysts=7 (expected 7)
[OK] scene 17 | cysts=7 (expected 7)
[OK] scene 18 | cysts=7 (expected 7)
[OK] scene 19 | cysts=7 (expected 7)
[OK] scene 20 | cysts=7 (expected 7)
[OK] scene 21 | cysts=7 (expected 7)
[OK] scene 22 | cysts=7 (expected 7)
[OK] scene 23 | cysts=7 (expected 7)
[OK] scene 24 | cysts=7 (expected 7)
[OK] scene 25 | cysts=7 (expected 7)
[OK] scene 26 | cysts=7 (expected 7)
[

## Flagged-Scene Segmentation Diagnostics

In [ ]:
# -------------------------------
# Debug flagged scenes: verify cyst segmentation intermediates
# -------------------------------
fallback_scene_df = scene_stats_df.loc[
    scene_stats_df['threshold_variant'].astype(str) != 'primary',
    [
        'scene_index',
        'scene_name',
        'threshold_method',
        'threshold_variant',
        'threshold_value',
        'autotune_used',
        'trim_applied',
        'split_applied',
    ],
].copy()

split_scene_df = scene_stats_df.loc[
    scene_stats_df['split_applied'].astype(bool),
    [
        'scene_index',
        'scene_name',
        'n_cysts_detected',
        'expected_cysts',
        'threshold_variant',
        'autotune_used',
        'split_applied',
    ],
].copy()

trimmed_scene_df = scene_stats_df.loc[
    scene_stats_df['trim_applied'].astype(bool),
    [
        'scene_index',
        'scene_name',
        'threshold_method',
        'threshold_variant',
        'threshold_value',
        'autotune_used',
        'trim_applied',
        'split_applied',
    ],
].copy()

flagged_scene_df = scene_stats_df.loc[
    ~scene_stats_df['cyst_count_ok'].astype(bool),
    [
        'scene_index',
        'scene_name',
        'n_cysts_detected',
        'expected_cysts',
        'threshold_method',
        'threshold_variant',
        'threshold_value',
        'autotune_used',
        'trim_applied',
        'split_applied',
    ],
].copy()

if fallback_scene_df.empty:
    print('No scenes required an Otsu-scale fallback.')
else:
    print('Scenes that required an Otsu-scale fallback:')
    display(fallback_scene_df)

if trimmed_scene_df.empty:
    print('No scenes required primary-Otsu trimming.')
else:
    print('Scenes trimmed after primary Otsu to remove extra components:')
    display(trimmed_scene_df)

if split_scene_df.empty:
    print('No scenes required post-segmentation touching-cyst splits.')
else:
    print('Scenes corrected by post-segmentation touching-cyst splits:')
    display(split_scene_df)

if flagged_scene_df.empty:
    print('No flagged scenes. Segmentation counts match expectations for all scenes.')
else:
    print(f'Flagged scenes for inline review: {len(flagged_scene_df)}')
    display(flagged_scene_df)

if RENDER_FLAGGED_SCENE_SEGMENTATION_DEBUG:
    if flagged_scene_df.empty:
        debug_scene_indices = [int(SCENE_FOR_DEBUG)]
    else:
        debug_scene_indices = flagged_scene_df['scene_index'].astype(int).tolist()
        if int(SCENE_FOR_DEBUG) not in debug_scene_indices:
            debug_scene_indices = [int(SCENE_FOR_DEBUG)] + debug_scene_indices

    debug_scene_indices = list(dict.fromkeys(debug_scene_indices))
    print(f'Rendering segmentation debug figures for {len(debug_scene_indices)} scene(s): {debug_scene_indices}')

    source_lookup = source_scene_manifest_df.set_index('scene_index', drop=False)
    for scene_idx in debug_scene_indices:
        source_row_dbg = source_lookup.loc[int(scene_idx)]
        channels_dbg, scene_shape_dbg, scene_name_dbg = load_scene_channels_from_manifest_row(
            source_row_dbg,
            SCENE_CHANNEL_COLS,
        )

        expected_dbg = expected_cyst_count_for_scene(scene_idx)
        cyst_params_dbg = dict(CYST_PARAMS)
        cyst_params_dbg['target_count'] = expected_dbg
        if int(scene_idx) in SPLIT_DISABLE_SCENES:
            cyst_params_dbg['split_touching_components_if_under'] = False
        if int(scene_idx) in SPLIT_DISABLE_SCENES:
            cyst_params_dbg['split_touching_components_if_under'] = False
        cyst_labels_dbg, cyst_props_dbg, cyst_debug_dbg = segment_cysts_from_dapi(
            channels_dbg['dapi'], cyst_params_dbg
        )
        cyst_labels_dbg, manual_override_info_dbg = apply_manual_split_override(
            scene_idx, cyst_labels_dbg, channels_dbg['dapi'], expected_dbg, cyst_params_dbg
        )
        if manual_override_info_dbg is not None:
            cyst_props_dbg = measure.regionprops(cyst_labels_dbg)
            cyst_debug_dbg['split_applied'] = True
            cyst_debug_dbg['manual_split_override'] = manual_override_info_dbg

        plot_cyst_segmentation_debug_figure(
            channels=channels_dbg,
            cyst_labels=cyst_labels_dbg,
            cyst_props=cyst_props_dbg,
            cyst_debug=cyst_debug_dbg,
            scene_index=scene_idx,
            scene_name=scene_name_dbg,
            expected_cysts=expected_dbg,
        )
elif RENDER_ALL_SCENE_SEGMENTATION_DEBUG:
    print(f'Rendering segmentation debug figures for {len(scene_manifest_df)} scenes.')
    for _, source_row_dbg in source_scene_manifest_df.iterrows():
        scene_idx = int(source_row_dbg['scene_index'])
        channels_dbg, scene_shape_dbg, scene_name_dbg = load_scene_channels_from_manifest_row(
            source_row_dbg,
            SCENE_CHANNEL_COLS,
        )

        expected_dbg = expected_cyst_count_for_scene(scene_idx)
        cyst_params_dbg = dict(CYST_PARAMS)
        cyst_params_dbg['target_count'] = expected_dbg
        if int(scene_idx) in SPLIT_DISABLE_SCENES:
            cyst_params_dbg['split_touching_components_if_under'] = False
        if int(scene_idx) in SPLIT_DISABLE_SCENES:
            cyst_params_dbg['split_touching_components_if_under'] = False
        cyst_labels_dbg, cyst_props_dbg, cyst_debug_dbg = segment_cysts_from_dapi(
            channels_dbg['dapi'], cyst_params_dbg
        )
        cyst_labels_dbg, manual_override_info_dbg = apply_manual_split_override(
            scene_idx, cyst_labels_dbg, channels_dbg['dapi'], expected_dbg, cyst_params_dbg
        )
        if manual_override_info_dbg is not None:
            cyst_props_dbg = measure.regionprops(cyst_labels_dbg)
            cyst_debug_dbg['split_applied'] = True
            cyst_debug_dbg['manual_split_override'] = manual_override_info_dbg

        plot_cyst_segmentation_debug_figure(
            channels=channels_dbg,
            cyst_labels=cyst_labels_dbg,
            cyst_props=cyst_props_dbg,
            cyst_debug=cyst_debug_dbg,
            scene_index=scene_idx,
            scene_name=scene_name_dbg,
            expected_cysts=expected_dbg,
        )
else:
    print('Skipping per-scene segmentation debug figures.')


No scenes required an Otsu-scale fallback.
Scenes trimmed after primary Otsu to remove extra components:


    scene_index                 scene_name threshold_method threshold_variant  \
29           29  well3-36locations.czi #30             otsu           primary   

    threshold_value  autotune_used  trim_applied  
29       430.470703          False          True  

Skipping per-scene segmentation debug figures (RENDER_ALL_SCENE_SEGMENTATION_DEBUG=False).


## Next Step
After reviewing the segmentation outputs for dataset 2, hand off to `<analysis-root>/pSMAD_2024-08-21/notebooks/02c_manual_bead_well_annotation.ipynb`.
